# ResumeIQ — Intelligent Resume Parser & Job Matching System
**Goal:** Parse a PDF resume, extract useful candidate information, compare it with a job description, and identify matched/missing skills.

**Approach:** PDF text extraction → rule-based entity/skill extraction → TF-IDF similarity + skill coverage → explainable results.

> This is a baseline project, not an automated hiring decision system. Scores are indicative and should not be used as the sole basis for hiring.

## 1. Install dependencies
Run this cell once. If Jupyter asks for a restart after installation, restart the kernel and continue.

In [ ]:
%pip install -q pymupdf pandas numpy scikit-learn matplotlib seaborn

## 2. Imports

In [ ]:
import re
import json
import textwrap
from pathlib import Path

import fitz  # PyMuPDF
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

## 3. Upload a resume PDF
Choose a resume PDF from your computer. Scanned/image-only PDFs may need OCR; this baseline extracts selectable text.

In [ ]:
from pathlib import Path

RESUME_PATH = input("Enter the full path to your resume PDF: ").strip().strip('"')
resume_path = Path(RESUME_PATH).expanduser()

if not resume_path.exists():
    raise FileNotFoundError(f"File not found: {resume_path}")
if resume_path.suffix.lower() != ".pdf":
    raise ValueError("Please provide a PDF resume.")

print("Resume:", resume_path.name)

## 4. Extract text from PDF

In [ ]:
def extract_pdf_text(pdf_path):
    text_parts = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc, start=1):
            page_text = page.get_text("text")
            if page_text.strip():
                text_parts.append(f"\n--- PAGE {page_num} ---\n{page_text}")
    return "\n".join(text_parts).strip()

resume_text = extract_pdf_text(resume_path)
print("Characters extracted:", len(resume_text))
print("\nPreview:\n")
print(resume_text[:4000] if resume_text else "No text extracted. This may be a scanned PDF.")

## 5. Provide a job description
Paste a job description below. You can replace the sample with a real job description for testing.

In [ ]:
job_description = '''
Machine Learning Engineer

Responsibilities:
- Build and evaluate machine learning models.
- Develop data pipelines and deploy ML services.
- Work with Python, SQL, pandas, scikit-learn, and Git.
- Communicate results and collaborate with engineering teams.

Requirements:
- Strong Python programming.
- Experience with machine learning and data preprocessing.
- Familiarity with NLP, model evaluation, and REST APIs.
- Bachelor's degree in Computer Science or a related field.
'''.strip()

print(job_description)

## 6. Clean text

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9+#.\s@:/_-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

resume_clean = clean_text(resume_text)
job_clean = clean_text(job_description)

print("Resume cleaned characters:", len(resume_clean))
print("Job description cleaned characters:", len(job_clean))

## 7. Extract common resume fields
These use transparent patterns and may need adjustment for different resume formats. Do not assume every field will be found.

In [ ]:
def first_match(patterns, text, flags=re.IGNORECASE):
    for pattern in patterns:
        match = re.search(pattern, text, flags)
        if match:
            return match.group(1).strip(" \t\n,;|")
    return None

def extract_email(text):
    match = re.search(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", text, re.I)
    return match.group(0) if match else None

def extract_phone(text):
    candidates = re.findall(r"(?:\+?\d[\d ()-]{8,}\d)", text)
    for candidate in candidates:
        digits = re.sub(r"\D", "", candidate)
        if 10 <= len(digits) <= 15:
            return candidate.strip()
    return None

def extract_links(text):
    urls = re.findall(r"(?:https?://)?(?:www\.)?(?:linkedin\.com|github\.com)/[^\s|,;]+", text, re.I)
    return list(dict.fromkeys(url.rstrip(").,") for url in urls))

candidate_info = {
    "email": extract_email(resume_text),
    "phone": extract_phone(resume_text),
    "links": extract_links(resume_text),
}
pd.DataFrame([candidate_info])

## 8. Skills dictionary
Start with a curated vocabulary. Expand it for the job families you care about. Phrase matching avoids treating every word as a skill.

In [ ]:
SKILLS = [
    "python", "java", "c++", "c", "sql", "javascript", "typescript",
    "machine learning", "deep learning", "natural language processing", "nlp",
    "generative ai", "large language models", "llm", "prompt engineering",
    "computer vision", "pytorch", "tensorflow", "scikit-learn", "pandas",
    "numpy", "matplotlib", "seaborn", "data analysis", "data visualization",
    "statistics", "feature engineering", "model evaluation", "classification",
    "regression", "clustering", "recommendation systems", "transformers",
    "hugging face", "langchain", "rag", "faiss", "vector databases",
    "fastapi", "flask", "streamlit", "rest api", "api development",
    "html", "css", "react", "node.js", "express", "git", "github",
    "postgresql", "mysql", "mongodb", "docker", "aws", "azure", "gcp",
    "data structures", "algorithms", "problem solving", "excel", "power bi",
    "tableau", "etl", "data preprocessing", "communication", "teamwork"
]

def find_skills(text, skill_list=SKILLS):
    normalized = clean_text(text)
    found = []
    for skill in skill_list:
        skill_norm = clean_text(skill)
        pattern = r"(?<![a-z0-9])" + re.escape(skill_norm) + r"(?![a-z0-9])"
        if re.search(pattern, normalized):
            found.append(skill)
    return sorted(set(found), key=str.lower)

resume_skills = find_skills(resume_text)
job_skills = find_skills(job_description)

print("Resume skills:", resume_skills)
print("\nSkills found in job description:", job_skills)

## 9. Match and missing skills

In [ ]:
matched_skills = sorted(set(resume_skills) & set(job_skills), key=str.lower)
missing_skills = sorted(set(job_skills) - set(resume_skills), key=str.lower)
extra_skills = sorted(set(resume_skills) - set(job_skills), key=str.lower)

skill_coverage = (len(matched_skills) / len(set(job_skills)) * 100) if job_skills else 0

print(f"Skill coverage: {skill_coverage:.1f}%")
print("Matched:", matched_skills)
print("Potential gaps:", missing_skills)
print("Other resume skills:", extra_skills)

## 10. TF-IDF text similarity
This compares the resume and job description using word and short-phrase overlap. It is a baseline relevance signal, not a probability of getting hired.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_features=10000)
tfidf = vectorizer.fit_transform([resume_clean, job_clean])
similarity = float(cosine_similarity(tfidf[0:1], tfidf[1:2])[0, 0])
text_similarity_pct = similarity * 100

print(f"TF-IDF cosine similarity: {text_similarity_pct:.2f}%")

## 11. Combine signals into an explainable baseline score
The weights are heuristic and should be tuned on a labeled evaluation set. This is not a calibrated hiring probability.

In [ ]:
# A transparent heuristic: 60% skill coverage + 40% text similarity.
# If no job skills are detected, use text similarity alone.
if job_skills:
    overall_match_score = 0.60 * skill_coverage + 0.40 * text_similarity_pct
else:
    overall_match_score = text_similarity_pct

result = {
    "resume_file": resume_path.name,
    "email": candidate_info["email"],
    "phone": candidate_info["phone"],
    "links": candidate_info["links"],
    "resume_skills": resume_skills,
    "job_skills": job_skills,
    "matched_skills": matched_skills,
    "missing_skills": missing_skills,
    "extra_skills": extra_skills,
    "skill_coverage_percent": round(skill_coverage, 2),
    "tfidf_similarity_percent": round(text_similarity_pct, 2),
    "overall_match_score_percent": round(overall_match_score, 2),
}

summary = pd.DataFrame([{
    "Skill coverage (%)": round(skill_coverage, 2),
    "TF-IDF similarity (%)": round(text_similarity_pct, 2),
    "Overall baseline score (%)": round(overall_match_score, 2),
    "Matched skills": len(matched_skills),
    "Potential skill gaps": len(missing_skills),
}])
summary

## 12. Visualize the matching signals

In [ ]:
labels = ["Skill coverage", "TF-IDF similarity", "Baseline score"]
values = [skill_coverage, text_similarity_pct, overall_match_score]

plt.figure(figsize=(8, 4))
bars = plt.bar(labels, values)
plt.ylim(0, 100)
plt.ylabel("Score (%)")
plt.title("Resume-to-Job Match Signals")
plt.xticks(rotation=10)
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f"{value:.1f}%", ha="center")
plt.tight_layout()
plt.show()

## 13. Export results

In [ ]:
output_path = Path("resume_match_results.json")
with output_path.open("w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

skills_table = pd.DataFrame({
    "Skill": sorted(set(job_skills) | set(resume_skills), key=str.lower)
})
skills_table["In resume"] = skills_table["Skill"].isin(resume_skills)
skills_table["Required by job description"] = skills_table["Skill"].isin(job_skills)
skills_table["Status"] = np.select(
    [
        skills_table["In resume"] & skills_table["Required by job description"],
        ~skills_table["In resume"] & skills_table["Required by job description"],
        skills_table["In resume"] & ~skills_table["Required by job description"],
    ],
    ["Matched", "Potential gap", "Additional resume skill"],
    default="Not identified"
)

skills_table.to_csv("skill_comparison.csv", index=False)
print("Saved:", output_path.resolve())
print("Saved:", Path("skill_comparison.csv").resolve())
skills_table

## 14. Sanity checks and limitations
- If the extracted text is empty, the PDF may be scanned and require OCR.
- Regex contact extraction can miss unusual formats; verify extracted details.
- The skills dictionary is incomplete and can miss synonyms or context.
- TF-IDF measures lexical overlap; it does not understand meaning as well as a semantic embedding model.
- The score is an explainable prototype metric, not a validated hiring prediction.
- Avoid using sensitive personal attributes or protected characteristics for ranking.